In [10]:
import pandas as pd

mises_papers = pd.read_csv("../data/raw/scopus_papers_citing_mises.csv")
hayek_papers = pd.read_csv("../data/raw/scopus_papers_citing_hayek.csv")

In [11]:
mises_papers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7458 entries, 0 to 7457
Data columns (total 23 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Authors            7439 non-null   object
 1   Author full names  7439 non-null   object
 2   Author(s) ID       7439 non-null   object
 3   Title              7458 non-null   object
 4   Year               7458 non-null   int64 
 5   Source title       7458 non-null   object
 6   Volume             4832 non-null   object
 7   Issue              4164 non-null   object
 8   Art. No.           404 non-null    object
 9   Page start         6913 non-null   object
 10  Page end           6908 non-null   object
 11  Cited by           7458 non-null   int64 
 12  DOI                6262 non-null   object
 13  Link               7458 non-null   object
 14  Abstract           7458 non-null   object
 15  Author Keywords    3669 non-null   object
 16  Index Keywords     1365 non-null   object


In [12]:
hayek_papers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19998 entries, 0 to 19997
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Authors            19908 non-null  object
 1   Author full names  19908 non-null  object
 2   Author(s) ID       19908 non-null  object
 3   Title              19998 non-null  object
 4   Year               19998 non-null  int64 
 5   Source title       19998 non-null  object
 6   Cited by           19998 non-null  int64 
 7   DOI                16555 non-null  object
 8   Link               19998 non-null  object
 9   Abstract           19998 non-null  object
 10  Author Keywords    9344 non-null   object
 11  Index Keywords     3513 non-null   object
 12  References         19998 non-null  object
 13  Open Access        4231 non-null   object
dtypes: int64(2), object(12)
memory usage: 2.1+ MB


In [13]:
# Adiciona flags antes de unificar
mises_papers['cite_mises'] = True
hayek_papers['cite_hayek'] = True

# Merge outer por Title + Source title, trazendo References do hayek também
all_papers = pd.merge(
    mises_papers,
    hayek_papers[['Title', 'Source title', 'cite_hayek', 'References', 'Author Keywords', 'Index Keywords']],
    on=['Title', 'Source title'],
    how='outer',
    suffixes=('', '_hayek')
)

# Garante References preenchido: prioriza o do mises, cai pro hayek se vazio
all_papers['References'] = all_papers['References'].combine_first(all_papers['References_hayek'])
all_papers = all_papers.drop(columns=['References_hayek'])

# Garante Author Keywords preenchido: prioriza o do mises, cai pro hayek se vazio
all_papers['Author Keywords'] = all_papers['Author Keywords'].combine_first(all_papers['Author Keywords_hayek'])
all_papers = all_papers.drop(columns=['Author Keywords_hayek'])

# Garante Author Keywords preenchido: prioriza o do mises, cai pro hayek se vazio
all_papers['Index Keywords'] = all_papers['Index Keywords'].combine_first(all_papers['Index Keywords_hayek'])
all_papers = all_papers.drop(columns=['Index Keywords_hayek'])


# Preenche os ausentes com False
all_papers['cite_mises'] = all_papers['cite_mises'].fillna(False)
all_papers['cite_hayek'] = all_papers['cite_hayek'].fillna(False)

total      = len(all_papers)
only_mises = (all_papers['cite_mises'] & ~all_papers['cite_hayek']).sum()
only_hayek = (all_papers['cite_hayek'] & ~all_papers['cite_mises']).sum()
ambos      = (all_papers['cite_mises'] &  all_papers['cite_hayek']).sum()
n_mises    = all_papers['cite_mises'].sum()
n_hayek    = all_papers['cite_hayek'].sum()

p_hayek_dado_mises = ambos / n_mises
p_mises_dado_hayek = ambos / n_hayek

print(f"mises_refs: {len(mises_papers):,} linhas")
print(f"hayek_refs: {len(hayek_papers):,} linhas")
print(f"all_refs:   {total:,} linhas")

print(f"\n{'Grupo':<20} {'N':>7} {'%':>7}")
print("-" * 36)
print(f"{'cite_mises only':<20} {only_mises:>7,} {only_mises/total*100:>6.1f}%")
print(f"{'cite_hayek only':<20} {only_hayek:>7,} {only_hayek/total*100:>6.1f}%")
print(f"{'ambos':<20} {ambos:>7,} {ambos/total*100:>6.1f}%")
print("-" * 36)
print(f"{'Total':<20} {total:>7,} {'100.0%':>7}")

print(f"\n{'Probabilidades condicionais'}")
print("-" * 36)
print(f"P(Hayek | Mises): {ambos:,} / {n_mises:,} = {p_hayek_dado_mises*100:.1f}%")
print(f"P(Mises | Hayek): {ambos:,} / {n_hayek:,} = {p_mises_dado_hayek*100:.1f}%")

# Checagem
vazios = all_papers['References'].isna().sum()
print(f"\nReferences vazias após merge: {vazios:,} ({vazios/total*100:.1f}%)")

mises_refs: 7,458 linhas
hayek_refs: 19,998 linhas
all_refs:   24,779 linhas

Grupo                      N       %
------------------------------------
cite_mises only        4,735   19.1%
cite_hayek only       17,272   69.7%
ambos                  2,772   11.2%
------------------------------------
Total                 24,779  100.0%

Probabilidades condicionais
------------------------------------
P(Hayek | Mises): 2,772 / 7,507 = 36.9%
P(Mises | Hayek): 2,772 / 20,044 = 13.8%

References vazias após merge: 0 (0.0%)


C:\Users\pedro\AppData\Local\Temp\ipykernel_18488\3457566548.py:28: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  all_papers['cite_mises'] = all_papers['cite_mises'].fillna(False)
C:\Users\pedro\AppData\Local\Temp\ipykernel_18488\3457566548.py:29: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  all_papers['cite_hayek'] = all_papers['cite_hayek'].fillna(False)


In [14]:
all_papers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24779 entries, 0 to 24778
Data columns (total 25 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Authors            7488 non-null   object 
 1   Author full names  7488 non-null   object 
 2   Author(s) ID       7488 non-null   object 
 3   Title              24779 non-null  object 
 4   Year               7507 non-null   float64
 5   Source title       24779 non-null  object 
 6   Volume             4839 non-null   object 
 7   Issue              4164 non-null   object 
 8   Art. No.           404 non-null    object 
 9   Page start         6952 non-null   object 
 10  Page end           6947 non-null   object 
 11  Cited by           7507 non-null   float64
 12  DOI                6293 non-null   object 
 13  Link               7507 non-null   object 
 14  Abstract           7507 non-null   object 
 15  Author Keywords    11719 non-null  object 
 16  Index Keywords     454

In [15]:
import pandas as pd

# tamanho original
n_total = len(all_papers)

# filtrar (mantém quem cita pelo menos um)
filtered_refs = all_papers[(all_papers['cite_mises']) | (all_papers['cite_hayek'])]

# tamanho depois
n_filtered = len(filtered_refs)

# quantidade removida
n_removed = n_total - n_filtered

# porcentagem removida
pct_removed = (n_removed / n_total) * 100

print(f"Removidos: {n_removed}")
print(f"% removida: {pct_removed:.2f}%")

Removidos: 0
% removida: 0.00%


In [16]:
import pandas as pd

# Garantir booleanos (caso estejam como 0/1)
all_papers['cite_mises'] = all_papers['cite_mises'].astype(bool)
all_papers['cite_hayek'] = all_papers['cite_hayek'].astype(bool)

# Criar indicador de NA em Author Keywords
all_papers['kw_nan'] = all_papers['Author Keywords'].isna()

# Agrupar por combinações de citação
result = (
    all_papers
    .groupby(['cite_mises', 'cite_hayek'])['kw_nan']
    .mean()  # proporção de NaN
    .reset_index()
)

# Converter para porcentagem
result['pct_kw_nan'] = result['kw_nan'] * 100

# (opcional) organizar melhor
result = result[['cite_mises', 'cite_hayek', 'pct_kw_nan']]

print(result)

   cite_mises  cite_hayek  pct_kw_nan
0       False        True   53.392774
1        True       False   49.862724
2        True        True   53.282828


Adding Abstract Keywords

In [17]:
!pip install pyahocorasick tqdm

In [18]:
import pandas as pd
import ahocorasick
from tqdm import tqdm

tqdm.pandas()

# Média de keywords ANTES da expansão
def count_keywords(kw_string):
    if pd.isna(kw_string) or kw_string == '':
        return 0
    return len([kw for kw in kw_string.split(';') if kw.strip()])

media_antes = all_papers['Author Keywords'].progress_apply(count_keywords).mean()
print(f"Média de keywords por artigo ANTES: {media_antes:.2f}")

# Coleta todas as keywords únicas
all_keywords = set()
for kw_string in tqdm(all_papers['Author Keywords'].dropna(), desc="Coletando keywords"):
    for kw in kw_string.split(';'):
        kw = kw.strip().lower()
        if kw and len(kw) > 3:
            all_keywords.add(kw)

print(f"Total de keywords únicas no dataset (>3 chars): {len(all_keywords):,}")

# 🔥 Constrói automato Aho-Corasick
automaton = ahocorasick.Automaton()

for idx, kw in enumerate(all_keywords):
    automaton.add_word(kw, kw)

automaton.make_automaton()

added_total = 0

def expand_keywords_fast(row):
    global added_total

    title    = row['Title']    if pd.notna(row['Title'])    else ''
    abstract = row['Abstract'] if pd.notna(row['Abstract']) else ''
    text = f"{title} {abstract}".lower()

    if not text:
        return row['Author Keywords']

    # Keywords existentes
    existing = set()
    if pd.notna(row['Author Keywords']) and row['Author Keywords'] != '':
        existing = {kw.strip().lower() for kw in row['Author Keywords'].split(';')}

    found = set()

    # 🔥 Busca eficiente
    for end_idx, kw in automaton.iter(text):
        # Checagem de boundary (simula \b do regex)
        start_idx = end_idx - len(kw) + 1

        before = text[start_idx - 1] if start_idx > 0 else ' '
        after  = text[end_idx + 1] if end_idx + 1 < len(text) else ' '

        if not before.isalnum() and not after.isalnum():
            found.add(kw)

    new_kws = found - existing

    if not new_kws:
        return row['Author Keywords']

    added_total += len(new_kws)

    base = row['Author Keywords'] if pd.notna(row['Author Keywords']) else ''
    new_kws_str = '; '.join(new_kws)

    return f"{base}; {new_kws_str}" if base else new_kws_str

# 🚀 Aplicação com progresso
all_papers['Abstract Keywords'] = all_papers.progress_apply(expand_keywords_fast, axis=1)

print(f"Total de keywords adicionadas: {added_total:,}")

100%|██████████| 24779/24779 [00:00<00:00, 397347.69it/s]


Média de keywords por artigo ANTES: 2.66


Coletando keywords: 100%|██████████| 11719/11719 [00:00<00:00, 225444.78it/s]


Total de keywords únicas no dataset (>3 chars): 28,818


100%|██████████| 24779/24779 [00:02<00:00, 11561.55it/s]

Total de keywords adicionadas: 298,725


Cleaning Journal Titles

In [19]:
# limpeza
all_papers['Source title'] = all_papers['Source title'].replace(
    "The Review of Austrian Economics",
    "Review of Austrian Economics"
)


Saving Everything

In [20]:
all_papers = all_papers.drop(['Page start', 'Page end'], axis=1)

In [21]:
all_papers.to_csv("../data/processed/all_papers.csv")
all_papers.sample(n=100).to_csv("../data/processed/all_papers_short.csv")